# 1. INTRODUCTION

This project explores the YouTube Trending Videos dataset from Kaggle.

The dataset contains information about trending YouTube videos across multiple regions, including variables such as title, channel, category, publish time, tags, views, likes, dislikes, comment counts, and whether comments or ratings are disabled.

The objective of this analysis is to understand the patterns behind trending videos, identify content and engagement factors associated with higher visibility, and prepare a clean dataset for deeper exploratory analysis.

This notebook will focus on:
- understanding the dataset structure
- cleaning and transforming columns
- engineering useful features
- preparing the data for univariate, bivariate, and multivariate analysis

# 2. IMPORT LIBRARIES

We will use the following libraries in this project:

- `pandas` for loading, cleaning, transforming, and analyzing tabular data
- `numpy` for numerical operations such as percentiles and conditional logic
- `matplotlib.pyplot` for creating plots
- `seaborn` for statistical visualizations 

In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

sns.set_theme(style='whitegrid')
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

rcParams["figure.figsize"] = (12, 6)


# 3. LOAD DATASET + RENAME COLUMNS

The YouTube Trending dataset contains separate CSV files for each country/region.
Each CSV has the same column structure but different trending video data.

We will:
1. Load multiple country CSVs and combine them into one DataFrame
2. Add a `country` column to track which region each row belongs to
3. Rename messy column names to clean snake_case
4. Parse `trending_date` and `publish_time` as proper datetime columns

In [7]:
import os

folder_path = r"C:\Users\HP\Downloads\archive (7)"
os.listdir(folder_path)[:10]

['CAvideos.csv',
 'CA_category_id.json',
 'DEvideos.csv',
 'DE_category_id.json',
 'FRvideos.csv',
 'FR_category_id.json',
 'GBvideos.csv',
 'GB_category_id.json',
 'INvideos.csv',
 'IN_category_id.json']

In [9]:
file_path = os.path.join(folder_path, "INvideos.csv")

df = pd.read_csv(file_path)

# extra spaces remove  , lower small letter convert 
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

print("Shape:", df.shape)
df.head()
df.sample()

Shape: (37352, 16)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
30663,Rlmxohf1i_8,18.08.05,Bucket List (Marathi with English Subtitle) | Official Trailer | Madhuri Dixit Nene | 25th May,Dharma Productions,1,2018-05-04T10:48:27.000Z,"Madhuri dixit|""Madhuri Dixit Nene""|""karan johar""|""Bucket list""|""Bucket""|""Bollywood""|""marathi""|""Cinema""|""regional cin...",4752988,40843,2122,2265,https://i.ytimg.com/vi/Rlmxohf1i_8/default.jpg,False,False,False,"Make a wish and tick it off your bucket list! Join Madhura in her journey filled with laughter, joy, acceptance and ..."


# Dataset Overview 

In this section , we take a first structured look at the dataset
We will use:
- `head()` to preview the first few rows
- `shape` to check the number of rows and columns
- `info()` to inspect data types and missing values
- `describe()` to summarize the distribution of variables

In [10]:
print("Shape:", df.shape)
df.head()

Shape: (37352, 16)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,kzwfHumJyYc,17.14.11,Sharry Mann: Cute Munda ( Song Teaser) | Parmish Verma | Releasing on 17 November,Lokdhun Punjabi,1,2017-11-12T12:20:39.000Z,"sharry mann|""sharry mann new song""|""sharry mann cute munda""|""sharry mann latest song""|""sharry mann punjabi song 2017...",1096327,33966,798,882,https://i.ytimg.com/vi/kzwfHumJyYc/default.jpg,False,False,False,Presenting Sharry Mann latest Punjabi Song Cute Munda Teaser . The music of new punjabi song is given by Gift Ruler...
1,zUZ1z7FwLc8,17.14.11,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दंग रह जायेंगे",HJ NEWS,25,2017-11-13T05:43:56.000Z,"पीरियड्स के समय|""पेट पर पति करता ऐसा""|""देखकर दंग रह जायेंगे""|""latest news""|""today news""|""news""|""breaking news""|""curr...",590101,735,904,0,https://i.ytimg.com/vi/zUZ1z7FwLc8/default.jpg,True,False,False,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दंग रह जायेंगे \n\nWatch this video :- https://youtu.be/zUZ1z7FwLc8\n\nH..."
2,10L1hZ9qa58,17.14.11,Stylish Star Allu Arjun @ ChaySam Wedding Reception | TFPC,TFPC,24,2017-11-12T15:48:08.000Z,"Stylish Star Allu Arjun @ ChaySam Wedding Reception|""Stylish Star Allu Arjun""|""ChaySam Wedding Reception""|""nagachait...",473988,2011,243,149,https://i.ytimg.com/vi/10L1hZ9qa58/default.jpg,False,False,False,"Watch Stylish Star Allu Arjun @ ChaySam Wedding Reception \n\n☛ For latest news https://www.tfpc.in, https://goo.gl..."
3,N1vE8iiEg64,17.14.11,Eruma Saani | Tamil vs English,Eruma Saani,23,2017-11-12T07:08:48.000Z,"Eruma Saani|""Tamil Comedy Videos""|""Films""|""Movies""|""Harija""|""Short""|""Film""|""Eruma Sani""|""Eruma Saani Videos""|""Eruma ...",1242680,70353,1624,2684,https://i.ytimg.com/vi/N1vE8iiEg64/default.jpg,False,False,False,This video showcases the difference between people who speak in English and Tamil in a very funny way.\nSubscribe to...
4,kJzGH0PVQHQ,17.14.11,why Samantha became EMOTIONAL @ Samantha naga chaithanya marriage Reception | Filmylooks,Filmylooks,24,2017-11-13T01:14:16.000Z,"Filmylooks|""latest news""|""telugu movies""|""telugu news""|""Tollywood news""|""why Samantha became EMOTIONAL @ Samantha na...",464015,492,293,66,https://i.ytimg.com/vi/kJzGH0PVQHQ/default.jpg,False,False,False,why Samantha became EMOTIONAL @ Samantha naga chaithanya marriage Reception | Filmylooks #samantha #Nagachaithanya \...


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37352 entries, 0 to 37351
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   video_id                37352 non-null  object
 1   trending_date           37352 non-null  object
 2   title                   37352 non-null  object
 3   channel_title           37352 non-null  object
 4   category_id             37352 non-null  int64 
 5   publish_time            37352 non-null  object
 6   tags                    37352 non-null  object
 7   views                   37352 non-null  int64 
 8   likes                   37352 non-null  int64 
 9   dislikes                37352 non-null  int64 
 10  comment_count           37352 non-null  int64 
 11  thumbnail_link          37352 non-null  object
 12  comments_disabled       37352 non-null  bool  
 13  ratings_disabled        37352 non-null  bool  
 14  video_error_or_removed  37352 non-null  bool  
 15  de

In [14]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
video_id,37352,16307,#NAME?,511,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,37352,205,17.14.11,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,37352,16721,Mission: Impossible - Fallout (2018) - Official Trailer - Paramount Pictures,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,37352,1426,VikatanTV,284,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,"37,352.00",NaN,NaN,NaN,21.58,6.56,1.00,23.00,24.00,24.00,43.00
publish_time,37352,16339,2018-04-21T13:30:01.000Z,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,37352,12578,[none],1381,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,"37,352.00",NaN,NaN,NaN,"1,060,477.65","3,184,932.05","4,024.00","123,915.50","304,586.00","799,291.25","125,432,237.00"
likes,"37,352.00",NaN,NaN,NaN,"27,082.72","97,145.10",0.00,864.00,"3,069.00","13,774.25","2,912,710.00"
dislikes,"37,352.00",NaN,NaN,NaN,"1,665.08","16,076.17",0.00,108.00,326.00,"1,019.25","1,545,017.00"


# 5. BUSINESS QUESTIONS

This exploratory analysis should help answer the following business and content-performance questions.
These questions move from basic understanding to deeper engagement and content pattern analysis.

1. Which videos receive the highest number of views?
2. Which channels appear most frequently in the trending list?
3. Which content categories are most common in trending videos?
4. What is the typical distribution of views, likes, dislikes, and comment counts?
5. How skewed are engagement metrics, and are there extreme outliers?
6. How often are comments disabled, ratings disabled, or videos removed?
7. Do videos with comments enabled receive higher average engagement?
8. Do videos with ratings enabled receive higher likes and views on average?

9. Which categories generate the highest average views and likes?
10. Which channels drive the strongest average engagement per trending video?
11. Is there a relationship between views and other engagement metrics such as likes, dislikes, and comments?
12. Are recently published videos more likely to trend quickly?
13. Do videos with missing or weak metadata such as tags or description perform differently?
14. Can we segment videos into performance bands using buckets or quartiles?
15. Which engineered features are most useful for comparing high-performing vs low-performing trending videos?

# 6. DATA CLEANING + FEATURE ENGINEERING
## 6.1 Detect Missing Values, Duplicates, and Data Types

Before changing anything, we first inspect data quality.

At this stage, we want to answer:
- Which columns have missing values?
- Are there duplicate rows?
- Which columns have incorrect or inconvenient data types?
- Which columns may need transformation before analysis?


In [19]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

description    561
dtype: int64

In [18]:
print("Duplicate rows:", df.duplicated().sum())
print()
print(df.dtypes)

Duplicate rows: 4263

video_id                  object
trending_date             object
title                     object
channel_title             object
category_id                int64
publish_time              object
tags                      object
views                      int64
likes                      int64
dislikes                   int64
comment_count              int64
thumbnail_link            object
comments_disabled           bool
ratings_disabled            bool
video_error_or_removed      bool
description               object
dtype: object


##  Remove Duplicate Rows and Fix Date Data Types

We found duplicate rows and two date-related columns stored as text.

We will:
- remove exact duplicate rows
- convert `trending_date` to datetime
- convert `publish_time` to datetime

This makes the dataset more reliable and prepares it for time-based analysis later.

In [ ]:

df = df.drop_duplicates()

print("Shape after removing duplicates",df.shape)

(33089, 16)
Shape after removing duplicates (33089, 16)


In [ ]:
df['trending_date']= pd.to_datetime(df['trending_date'],format="%y.%d.%m",errors='coerce')
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")

# erros = 'coerce'  nat if ant error nat missing datetime value
print(df[["trending_date", "publish_time"]].dtypes)

trending_date         datetime64[ns]
publish_time     datetime64[ns, UTC]
dtype: object


## Handling Missing Values

only the description column has missing values 


In [29]:
print("Missing values")
df.isnull().sum()

Missing values


video_id                    0
trending_date               0
title                       0
channel_title               0
category_id                 0
publish_time                0
tags                        0
views                       0
likes                       0
dislikes                    0
comment_count               0
thumbnail_link              0
comments_disabled           0
ratings_disabled            0
video_error_or_removed      0
description               527
dtype: int64

In [30]:
df['description'] = df['description'].fillna("No description")
print("Missing values after cleaning")
print(df.isnull().sum())

Missing values after cleaning
video_id                  0
trending_date             0
title                     0
channel_title             0
category_id               0
publish_time              0
tags                      0
views                     0
likes                     0
dislikes                  0
comment_count             0
thumbnail_link            0
comments_disabled         0
ratings_disabled          0
video_error_or_removed    0
description               0
dtype: int64


# Create a Binary Feature with 
`np.where()`

We will create a new column called `has_description`.

This feature is useful because:
- the original `description` column is long text and not easy to compare directly
- a simple 0/1 flag helps us later test whether videos with a description perform differently
- this fits the dataset well because we already filled missing descriptions with `No description

In [58]:
df['has_description']=np.where(df['description'] == "No description",0,1 )
# 0 - no description , 1 - description
df[['description','has_description']].sample(5)

,description,has_description
20635,No description,0
561,"Ghantakhanek sangesuman: Youth TMC Targets Mukul at Kolkata rally, Although Subhrangshu, Mukul's son remains absent ...",1
21157,கர்ப்பிணி பெண் உஷாவின் இறுதி ஊர்வலம் - பொதுமக்கள் கண்ணீர் அஞ்சலி | Usha | Trichy | Kamaraj | Thanthi TV | Usha Fune...,1
9399,బాలిక బ్రతికుండగానే చనిపోయింది అని చెప్పిన వైద్యులు || అంత్యక్రియల్లో కదిలిన బాలిక #9Roses Media\n\nThank you for Wa...,1
19239,நடிகை ஸ்ரீதேவியின் உடல் அவரது இல்லத்திற்கு கொண்டுவரப்பட்டது\n\nConnect with Puthiya Thalaimurai TV Online:\n\nSUBSCR...,1


In [54]:
print('Orignal description is has long column name so its diffiuclt to compare so has_description')

Orignal description is has long column name so its diffiuclt to compare so has_description


In [ ]:
df['has_description'].value_counts()
(df['has_description'].value_counts(normalize=True).mul(100).round(2))

# percentage how many 0 and 1

has_description
1   98.41
0    1.59
Name: proportion, dtype: float64

## Create Time-Based Features

The dataset contains two important date columns:
- `publish_time` = when the video was uploaded
- `trending_date` = when the video appeared in the trending list
This columns are useful for feature enginnering because timing can influence visibility and engagement
- publish date(only keep upload date) , days to trend(how qucikyl a video reached trending), publish hour , publish weekday(to identify the upload day name )

- remove timezone -.dt.tz_localize(None) +00.00 .dt.normalize date only 
- 02 - trending_date-publish_date con to dt.days - days take trend
- 03 -dt.tz_localize(None).dt.hour - hour extract
- 04- publish_weekday --  publishh time timezone remove and add.dt.day_name()


In [ ]:
# publish date - In publish time full  timestamp required date
df['publish_date']= df['publish_time'].dt.tz_localize(None).dt.normalize()

df['publish_date'].head(3)
# After uploading the video how much it take time to in trend
df['days_to_trend']= (df['trending_date']-df['publish_date']).dt.days

df['publish_hour']= df['publish_time'].dt.tz_localize(None).dt.hour
# which day uploaded - publish weekday
df["publish_weekday"] = df["publish_time"].dt.tz_localize(None).dt.day_name()

df[["publish_time", "trending_date", "publish_date", "days_to_trend", "publish_hour", "publish_weekday"]].head()


,publish_time,trending_date,publish_date,days_to_trend,publish_hour,publish_weekday
0,2017-11-12 12:20:39+00:00,2017-11-14,2017-11-12,2,12,Sunday
1,2017-11-13 05:43:56+00:00,2017-11-14,2017-11-13,1,5,Monday
2,2017-11-12 15:48:08+00:00,2017-11-14,2017-11-12,2,15,Sunday
3,2017-11-12 07:08:48+00:00,2017-11-14,2017-11-12,2,7,Sunday
4,2017-11-13 01:14:16+00:00,2017-11-14,2017-11-13,1,1,Monday


In [64]:
print(df["days_to_trend"].describe())
print()
print(df["publish_weekday"].value_counts())

count   33,089.00
mean         2.11
std          2.16
min          0.00
25%          1.00
50%          2.00
75%          3.00
max        221.00
Name: days_to_trend, dtype: float64

publish_weekday
Friday       5573
Saturday     5196
Thursday     4986
Tuesday      4619
Monday       4607
Wednesday    4550
Sunday       3558
Name: count, dtype: int64


## Create a Derived Category with `apply()`

The `tags` column is not directly analysis-friendly because it contains long text separated by `|`, and some videos use `[none]`.

A custom function fits this column well because we want rule-based classification:
- `No Tags` if tags are missing or `[none]`
- `Low Tag Use` if only a few tags are used
- `Medium Tag Use` for moderate tag count
- `High Tag Use` for heavy tag usage

This gives us a simpler category for later comparison.

tag_text.split - music|pop|india split into list

In [ ]:
# NEW COLUMN CREATE 
def tag(tag_text):
    if pd.isna(tag_text) or tag_text == 'None':
       return "No Tags"
    tag_count = len(tag_text.split("|"))

    if tag_count <=5:
        return "Low Tag use"
    elif tag_count <15:
        return "Medium Tag use"
    else:
        return "High Tag use"

# new column create
# category high low medium
df['tag_use_level'] = df['tags'].apply(tag)
df[["tags", "tag_use_level"]].head()

,tags,tag_use_level
0,"sharry mann|""sharry mann new song""|""sharry mann cute munda""|""sharry mann latest song""|""sharry mann punjabi song 2017...",High Tag use
1,"पीरियड्स के समय|""पेट पर पति करता ऐसा""|""देखकर दंग रह जायेंगे""|""latest news""|""today news""|""news""|""breaking news""|""curr...",High Tag use
2,"Stylish Star Allu Arjun @ ChaySam Wedding Reception|""Stylish Star Allu Arjun""|""ChaySam Wedding Reception""|""nagachait...",Medium Tag use
3,"Eruma Saani|""Tamil Comedy Videos""|""Films""|""Movies""|""Harija""|""Short""|""Film""|""Eruma Sani""|""Eruma Saani Videos""|""Eruma ...",High Tag use
4,"Filmylooks|""latest news""|""telugu movies""|""telugu news""|""Tollywood news""|""why Samantha became EMOTIONAL @ Samantha na...",Medium Tag use


In [68]:
print(df["tag_use_level"].value_counts())
print()
print(df["tag_use_level"].value_counts(normalize=True).mul(100).round(2))

tag_use_level
High Tag use      21120
Medium Tag use     9252
Low Tag use        2717
Name: count, dtype: int64

tag_use_level
High Tag use     63.83
Medium Tag use   27.96
Low Tag use       8.21
Name: proportion, dtype: float64
